# 🔍 Lakeflow CDC Source Validator — Phase 1
### DNS · TCP · TLS · Optional App Probe

Validates pre-flight connectivity for **Lakeflow CDC database connectors** before pipeline deployment or troubleshooting.

| Source | Default Port | Notes |
|---|---|---|
| SQL Server | 1433 | Pure Python TDS protocol for app probe |
| PostgreSQL | 5432 | psycopg2-binary for app probe |
| MySQL | 3306 | mysql-connector-python for app probe |

> **Drivers:** The next cell auto-installs `psycopg2-binary` and `mysql-connector-python` for the optional app probes. SQL Server uses a built-in pure-Python TDS implementation.

**How to run:** Fill in the widgets at the top of the page, then click **Run All**.

**Credentials:** The app probe (optional) reads the DB password from a **Unity Catalog secret** at runtime:

```python
db_pwd = dbutils.secrets.get(catalog="<catalog_name>", schema="<schema_name>", key="<secret_key>")
```

Fill in the `secret_catalog`, `secret_schema`, and `secret_key` widgets with the coordinates of your secret.

> **Note:** Creating UC secrets is currently supported only via the **UI** or the **REST API** — not SQL or the CLI. See [Create a secret](https://docs.databricks.com/aws/en/security/secrets/unity-catalog-secrets#create-a-secret) for instructions.

In [0]:
%pip install psycopg2-binary mysql-connector-python --quiet
dbutils.library.restartPython()

In [0]:
# ── Cell 1 · User Configuration ─────────────────────────────────────────────────────────────────────────────────────
# This cell defines the user-facing inputs for the validator.
#
# Why widgets?
# * They make the notebook easy to run manually from the UI.
# * They keep the rest of the notebook generic because later cells only read `config`.
#
# Typical use:
# * Set `host` and optionally `port`
# * Leave `port` blank to use the default from SOURCE_PROFILES
# * Turn on `run_app_probe` only when you want to test real DB login/query access
# * Provide `username` / `database` and the secret coordinates only for the optional app probe
#
# For scripted or automated runs, you can also override values directly in `config`.

# NOTE: do NOT removeAll() on re-run — it wipes values you typed into the
# widgets before clicking Run All. Re-declaring widgets below is idempotent
# and preserves existing values.

dbutils.widgets.dropdown(
    "source_type", "sqlserver",
    ["sqlserver", "postgres", "mysql"],
    "Source Type",
)
dbutils.widgets.text("host",             "",      "Source Hostname")
dbutils.widgets.text("port",             "",      "Port  (blank = profile default)")
dbutils.widgets.dropdown("tls_enabled",  "true",  ["true", "false"], "TLS Enabled")
dbutils.widgets.dropdown("run_app_probe","false", ["true", "false"], "Run App Probe")
dbutils.widgets.text("username",         "",      "Username  (app probe only)")
dbutils.widgets.text("secret_catalog",   "",      "Secret Catalog  (app probe only)")
dbutils.widgets.text("secret_schema",    "",      "Secret Schema  (app probe only)")
dbutils.widgets.text("secret_key",       "",      "Secret Key  (app probe only)")
dbutils.widgets.text("secret_scope",     "",      "Secret Scope  (legacy fallback, app probe only)")
dbutils.widgets.text("database",         "",      "Database  (app probe only)")
dbutils.widgets.text("timeout_seconds",  "5",     "Timeout (seconds)")


# ── Read widget values ─────────────────────────────────────────────────────────────────────────────────────────
def _w(key, default=""):
    """Read a widget value safely, returning ``default`` on any error."""
    try:
        v = dbutils.widgets.get(key)
        return v if v is not None else default
    except Exception:
        return default


# Normalize widget values into a single config dictionary.
# Downstream cells only depend on `config`, so they do not need to care whether
# the values came from widgets, defaults, or a future automation wrapper.
# ── Read password from Unity Catalog secret ────────────────────────────────────────────────────────────────
# The password is stored as a UC secret and retrieved at runtime.
# See: https://docs.databricks.com/aws/en/security/secrets/unity-catalog-secrets
_secret_catalog = (_w("secret_catalog", "") or "").strip() or None
_secret_schema  = (_w("secret_schema",  "") or "").strip() or None
_secret_key     = (_w("secret_key",     "") or "").strip() or None

_secret_scope   = (_w("secret_scope",   "") or "").strip() or None

if _secret_catalog and _secret_schema and _secret_key:
    # Unity Catalog secret (create via UI or REST API).
    _password = dbutils.secrets.get(
        catalog=_secret_catalog, schema=_secret_schema, key=_secret_key
    )
elif _secret_scope and _secret_key:
    # Legacy workspace secret scope (CLI/Terraform-creatable) — fallback.
    _password = dbutils.secrets.get(scope=_secret_scope, key=_secret_key)
else:
    _password = None

config = {
    "source_type":     (_w("source_type", "sqlserver") or "sqlserver").strip().lower(),
    "host":            (_w("host", "")).strip(),
    "port":            (_w("port", "") or "").strip() or None,
    "tls_enabled":      _w("tls_enabled",   "true")  == "true",
    "run_app_probe":    _w("run_app_probe",  "false") == "true",
    "username":         _w("username",       "") or None,
    "password":         _password,
    "secret_catalog":   _secret_catalog,
    "secret_schema":    _secret_schema,
    "secret_key":       _secret_key,
    "secret_scope":     _secret_scope,

    "database":         _w("database",       "") or None,
    "timeout_seconds":  int(_w("timeout_seconds", "5") or 5),
}

# Mask secrets before printing so the notebook is safe to demo or share.
_safe = {k: ("***" if k in ("password",) and v else v) for k, v in config.items()}
print("Config loaded →", _safe)


In [0]:
# ── Cell 2 · Source Profiles ────────────────────────────────────────────────────────────────────────────────────────────
# Central registry of per-engine defaults and metadata.
#
# Design intent:
# * Keep source-specific behavior isolated in one place.
# * Let the rest of the notebook stay generic.
# * Make future engines easy to add without changing the core validator logic.
#
# Each profile contains:
# * default_port  → used when the user leaves the port widget blank
# * tls_default   → default expectation for whether TLS should be used
# * display_name  → nicer label for human-readable output
# * probe_note    → installation hint if the optional driver-based probe is unavailable

SOURCE_PROFILES = {
    "sqlserver": {
        "default_port": 1433,
        "tls_default":  True,
        "display_name": "SQL Server",
        "probe_note":   "Pure Python TDS protocol (no extra install needed).",
    },
    "postgres": {
        "default_port": 5432,
        "tls_default":  True,
        "display_name": "PostgreSQL",
        "probe_note":   "Requires psycopg2  (pip install psycopg2-binary).",
    },
    "mysql": {
        "default_port": 3306,
        "tls_default":  True,
        "display_name": "MySQL",
        "probe_note":   "Requires mysql-connector-python  (pip install mysql-connector-python).",
    },
}

# Convenience set used for input validation in the runner.
VALID_SOURCE_TYPES = set(SOURCE_PROFILES.keys())
print(f"Source profiles loaded: {', '.join(sorted(SOURCE_PROFILES))}")


In [0]:
# ── Cell 3 · Core Utility Functions ──────────────────────────────────────────────────────────────────────────────────────────
# These helpers implement the reusable building blocks of the validator.
#
# The sequence mirrors the way connectivity issues are usually diagnosed:
# 1. DNS  → can the hostname be resolved?
# 2. TCP  → can the environment open a socket to host:port?
# 3. TLS  → if requested, can it complete an SSL/TLS handshake?
#
# Keeping each check separate makes the result easier to classify and explain.
import socket
import ssl
import time
import json
import errno as _errno
from typing import Any, Dict


# ── Shared helper ───────────────────────────────────────────────────────────────────────────────────────────────
def _result(status: str, details: str, **extra) -> Dict[str, Any]:
    """Build a normalized result object shared by every check.

    Using one schema everywhere makes it easy to:
    * print a summary
    * generate JSON output
    * add future export or reporting features
    """
    out: Dict[str, Any] = {"status": status, "details": details}
    out.update(extra)
    return out


# ── DNS ──────────────────────────────────────────────────────────────────────────────────────────────────────
def resolve_host(host: str) -> Dict[str, Any]:
    """Resolve hostname to one or more IP addresses.

    This is the earliest and cheapest check. If DNS fails, later network checks are
    usually not meaningful, so the runner will skip them.
    """
    if not host:
        return _result("FAIL", "No hostname provided. Fill in the 'host' widget.")
    start = time.monotonic()
    try:
        infos = socket.getaddrinfo(host, None)
        ips = sorted({item[4][0] for item in infos})
        latency_ms = round((time.monotonic() - start) * 1000, 2)
        return _result(
            "PASS",
            f"Resolved {host!r} → {ips}",
            ips=ips,
            latency_ms=latency_ms,
        )
    except socket.gaierror as e:
        return _result("FAIL", f"DNS resolution failed for {host!r}: {e}")
    except Exception as e:
        return _result("FAIL", f"Unexpected DNS error: {e}")


# ── TCP ──────────────────────────────────────────────────────────────────────────────────────────────────────
def tcp_check(
    host: str,
    port: int,
    timeout_seconds: int = 5,
    resolved_ips=None,
) -> Dict[str, Any]:
    """Attempt a bare TCP connection to host:port.

    This does not prove the database is usable yet; it only proves that something on
    the network path accepted a socket connection.

    resolved_ips: pass the IPs from the DNS check so error messages include the
    actual IP being dialed, making them more actionable during troubleshooting.
    """
    ip_hint = f" (resolved to {resolved_ips})" if resolved_ips else ""
    start = time.monotonic()
    try:
        with socket.create_connection((host, port), timeout=timeout_seconds):
            latency_ms = round((time.monotonic() - start) * 1000, 2)
            return _result(
                "PASS",
                f"TCP connection to {host}:{port} succeeded.",
                latency_ms=latency_ms,
            )

    except socket.timeout:
        elapsed_ms = round((time.monotonic() - start) * 1000, 2)
        # Timing signal: a full-duration timeout means the firewall is silently
        # dropping packets (no RST ever returned). A fast failure means something
        # close to this compute rejected the packet immediately.
        if elapsed_ms < timeout_seconds * 1000 * 0.5:
            timing_hint = (
                f"Failed quickly ({elapsed_ms} ms) — likely an immediate packet rejection "
                "by a firewall or NACL close to this compute."
            )
        else:
            timing_hint = (
                f"Timed out after the full {timeout_seconds}s — a firewall or security group "
                "is likely silently dropping packets (no TCP RST was returned)."
            )
        return _result(
            "FAIL",
            f"TCP timeout reaching {host}:{port}{ip_hint}. {timing_hint} "
            f"Check your database's firewall / security group inbound rules for port {port}, "
            "network ACLs, route tables, and whether the instance allows external connections.",
        )

    except ConnectionRefusedError:
        elapsed_ms = round((time.monotonic() - start) * 1000, 2)
        # Refused means the host responded — it is reachable — but nothing accepted
        # the connection on this port. Very different from a silent timeout.
        return _result(
            "FAIL",
            f"Connection actively refused at {host}:{port}{ip_hint} ({elapsed_ms} ms). "
            "The host is reachable but rejected the connection — "
            "verify the port is correct, that the database service is running, "
            "and that it is configured to accept remote connections.",
            latency_ms=elapsed_ms,
        )

    except OSError as e:
        elapsed_ms = round((time.monotonic() - start) * 1000, 2)
        err = e.errno
        # Inspect the OS errno to give a precise routing vs host vs timeout diagnosis.
        if err == _errno.ENETUNREACH:
            detail = (
                f"Network unreachable to {host}:{port}{ip_hint}. "
                "No route from this compute to the target network. "
                "Check VPC routing tables, VPN/Direct Connect attachment, "
                "or subnet route to IGW/TGW."
            )
        elif err == _errno.EHOSTUNREACH:
            detail = (
                f"Host unreachable at {host}:{port}{ip_hint}. "
                "A route to the network exists but this specific host is not responding. "
                "Check host-level firewall, whether the IP is correct, "
                "and whether the instance is running."
            )
        elif err == _errno.ETIMEDOUT:
            detail = (
                f"OS-level connection timeout to {host}:{port}{ip_hint}. "
                "A firewall or security group is likely silently dropping packets. "
                "Check the database's firewall / security group inbound rules and network ACLs."
            )
        else:
            detail = f"TCP connection to {host}:{port} failed (OS error {err}): {e}"
        return _result("FAIL", detail)

    except Exception as e:
        return _result("FAIL", f"Unexpected TCP error: {e}")


# ── TLS ──────────────────────────────────────────────────────────────────────────────────────────────────────
def _flatten_rdns(rdns_seq) -> str:
    """Convert cert subject/issuer RDN sequence to a readable string."""
    _short = {
        "commonName": "CN", "organizationName": "O",
        "countryName": "C", "localityName": "L",
        "stateOrProvinceName": "ST", "organizationalUnitName": "OU",
    }
    parts = []
    for rdn in rdns_seq:
        for attr, val in rdn:
            parts.append(f"{_short.get(attr, attr)}={val}")
    return ", ".join(parts) if parts else str(rdns_seq)


def tls_check(host: str, port: int, timeout_seconds: int = 5) -> Dict[str, Any]:
    """Attempt a full TLS handshake with certificate verification.

    Important nuance:
    * PASS = handshake succeeded and certificate verified
    * WARN = TLS endpoint is reachable, but certificate verification failed
    * FAIL = handshake could not complete at all

    That WARN/FAIL distinction is useful because it separates trust problems from
    deeper protocol or reachability issues.
    """
    context = ssl.create_default_context()
    try:
        with socket.create_connection((host, port), timeout=timeout_seconds) as raw:
            with context.wrap_socket(raw, server_hostname=host) as tls:
                cert    = tls.getpeercert()
                version = tls.version()
                cipher  = tls.cipher()          # (name, protocol, bits)
                subject = _flatten_rdns(cert.get("subject", ()))
                issuer  = _flatten_rdns(cert.get("issuer",  ()))
                san     = [v for _, v in cert.get("subjectAltName", ())]
                expiry  = cert.get("notAfter", "unknown")
                return _result(
                    "PASS",
                    f"TLS handshake succeeded ({version}, {cipher[0]}). "
                    f"Subject: {subject}. Expires: {expiry}.",
                    tls_version=version,
                    cipher=cipher[0],
                    subject=subject,
                    issuer=issuer,
                    san=san,
                    expiry=expiry,
                )
    except ssl.SSLCertVerificationError as e:
        return _result(
            "WARN",
            f"TLS is reachable but certificate verification failed: {e}. "
            "Check CA trust bundle, hostname/SAN match, or private CA requirements.",
        )
    except ssl.SSLError as e:
        return _result(
            "FAIL",
            f"TLS handshake failed: {e}. "
            "Check that TLS is enabled on the source and that the protocol/cipher is supported.",
        )
    except socket.timeout:
        return _result("FAIL", f"TCP timeout during TLS handshake to {host}:{port}.")
    except Exception as e:
        return _result("FAIL", f"TLS connection failed: {e}")


# ── Result aggregation ────────────────────────────────────────────────────────────────────────────────────────
def final_status(checks: Dict[str, Any]) -> str:
    """Aggregate individual check statuses into PASS / WARN / FAIL."""
    statuses = [v["status"] for v in checks.values()]
    if "FAIL" in statuses:
        return "FAIL"
    if "WARN" in statuses:
        return "WARN"
    return "PASS"


def classify_recommendation(checks: Dict[str, Any], host: str = "", port: int = 0, network_info: Dict[str, Any] | None = None) -> str:
    """Return a plain-language troubleshooting hint.

    Accepts host and port so it can detect cloud-specific patterns (e.g. RDS endpoints)
    and give a prioritised, ordered list of causes rather than a generic checklist.

    network_info: when provided, uses confirmed facts (e.g. Publicly Accessible = Yes)
    to eliminate irrelevant suggestions and avoid false positives.
    """
    network_info = network_info or {}
    dns = checks["dns"]["status"]
    tcp = checks["tcp"]["status"]
    tls = checks["tls"]["status"]
    app = checks["app_probe"]["status"]
    tcp_detail = checks["tcp"].get("details", "").lower()

    # Detect well-known endpoint patterns for more targeted advice.
    is_rds  = ".rds.amazonaws.com"   in host.lower()
    is_azure = ".database.windows.net" in host.lower() or ".database.azure.com" in host.lower()

    if dns == "FAIL":
        return (
            "Likely DNS issue. Check hostname spelling, private DNS zones, "
            "custom resolvers, or VPC/VNet DNS settings."
        )

    if tcp == "FAIL":
        if "refused" in tcp_detail:
            # Active refusal = host reachable, but service not accepting on this port.
            return (
                "TCP connection was actively refused — the host is reachable but the port "
                "is closed or the database service is not running. "
                "Verify the port is correct, that the DB service is started, "
                "and that it is configured to accept remote connections."
            )
        if "unreachable" in tcp_detail:
            return (
                "Network or host unreachable. Check VPC routing tables, VPN/Direct Connect "
                "attachment, and whether the target subnet has a return route to this compute."
            )

        # Silent timeout: narrow by endpoint type.
        if is_rds:
            # Check if we've confirmed Publicly Accessible from network_info
            rds_pa = network_info.get("rds_public_access", {})
            publicly_accessible_confirmed = rds_pa.get("publicly_accessible") is True

            steps = []
            step_num = 1
            if not publicly_accessible_confirmed:
                steps.append(
                    f"  {step_num}. Publicly Accessible = No — most likely cause. "
                    f"Open the RDS console → your instance → Connectivity & security. "
                    f"If 'Publicly accessible' is No, the instance only accepts connections "
                    f"from within its VPC. You need VPC peering, Transit Gateway, or "
                    f"PrivateLink to reach it from outside (including from this compute)."
                )
                step_num += 1
            steps.append(
                f"  {step_num}. Security group inbound rule missing — check the security group "
                f"attached to the RDS instance and confirm there is an inbound rule "
                f"allowing TCP {port} from this compute's IP or CIDR. "
                f"Note: on serverless compute, the egress IP can rotate from a shared pool. "
                f"Whitelist the Databricks serverless CIDR ranges from ip-ranges.json, not a single /32."
            )
            step_num += 1
            steps.append(
                f"  {step_num}. VPC NACL — less common but NACLs can silently drop traffic "
                f"even when the security group is open. Check the NACL on the RDS subnet."
            )

            header = f"Silent TCP timeout to RDS endpoint on port {port}."
            if publicly_accessible_confirmed:
                header += " 'Publicly Accessible' is confirmed YES (resolved to public IP), so that's ruled out."
            header += "\nCheck in this order:\n"

            return header + "\n".join(steps)
        if is_azure:
            return (
                f"Silent TCP timeout to Azure SQL/Postgres endpoint on port {port}. "
                f"Check in this order:\n"
                f"  1. Public network access — open the Azure portal → your server → "
                f"Networking. If 'Public network access' is Disabled, only private "
                f"endpoint connections are accepted.\n"
                f"  2. Firewall rules — confirm the client IP range is listed under "
                f"'Firewall rules' on the Networking blade.\n"
                f"  3. Private endpoint / VNet injection — if access is restricted to "
                f"a VNet, verify the peering or private endpoint is in place."
            )
        # Generic / unknown cloud silent drop.
        return (
            "Silent TCP timeout — a firewall or security group is dropping packets. "
            "Check the DB firewall / security group inbound rules for the target port, "
            "network ACLs, and route tables. If using a private subnet, verify that "
            "VPC peering, Transit Gateway, PrivateLink, or VPN is active."
        )

    if tls == "FAIL":
        return (
            "Likely TLS configuration issue. Verify TLS is enabled on the source, "
            "and that the protocol version and cipher suite are supported."
        )
    if tls == "WARN":
        return (
            "Likely TLS trust issue. Check CA trust bundle, certificate chain, "
            "hostname/SAN match, or private CA requirements."
        )
    if app == "FAIL":
        return (
            "Network path looks OK. Review credentials, database user permissions, "
            "auth mode (e.g. SQL vs Windows auth for SQL Server), or engine-specific "
            "CDC prerequisites."
        )
    return "Connectivity looks good for a Phase 1 pre-flight check."


# ── Human-readable summary printer ─────────────────────────────────────────────────────────────────────────────
STATUS_ICON = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌", "SKIPPED": "⏭️"}


def print_summary(result: Dict[str, Any]) -> None:
    """Print a human-readable summary for notebook users.

    The JSON result is better for automation; this summary is better for a quick
    read during manual validation and troubleshooting.
    """
    profile  = SOURCE_PROFILES.get(result["source_type"], {})
    display  = profile.get("display_name", result["source_type"])
    checks   = result["checks"]
    fs_icon  = STATUS_ICON.get(result["final_status"], "?")

    W = 64
    print("═" * W)
    print(f"  Lakeflow CDC Source Validator — Phase 1")
    print("═" * W)
    print(f"  Source  : {display}  ({result['host']}:{result['port']})")
    print("─" * W)
    for name, check in checks.items():
        icon = STATUS_ICON.get(check["status"], "?")
        lat  = f"  [{check['latency_ms']} ms]" if "latency_ms" in check else ""
        print(f"  {name.upper():<13}{icon} {check['status']:<8}{lat}")
        print(f"               {check['details']}")
    print("─" * W)
    print(f"  RESULT  : {fs_icon} {result['final_status']}")
    print(f"  HINT    : {result['recommendation']}")
    print("═" * W)


print("Core utility functions loaded.")


In [0]:
# ── Cell 4 · Optional App Probes ───────────────────────────────────────────────────────────────────────────────────────
# These checks are intentionally optional.
#
# Why optional?
# * DNS/TCP/TLS are enough for a useful pre-flight network test.
# * App probes require credentials and database drivers that may not be installed.
# * In early troubleshooting, you often want to separate network path issues from
#   auth and DB permission issues.
#
# Each probe attempts a tiny source-specific login + query to confirm that the
# validator can do more than just open a socket.
#
# Missing driver  → graceful SKIPPED result with an installation hint.
# Credentials are only used in these probes and are never logged.


# Shared helper used when the Python driver is not installed on the cluster.
def _probe_unavailable(source_type: str) -> Dict[str, Any]:
    note = SOURCE_PROFILES.get(source_type, {}).get("probe_note", "Driver not installed.")
    return _result(
        "SKIPPED",
        f"App probe driver unavailable. {note}",
    )


# ── SQL Server ────────────────────────────────────────────────────────────────────────────────────────────────
# SQL Server probe uses pure Python TDS (Tabular Data Stream) protocol.
#
# Why not ODBC or JDBC?
# * The Databricks Runtime does NOT include system-level ODBC drivers (msodbcsql17/18).
# * Spark JDBC (spark.read.format("jdbc")) on serverless compute cannot make
#   outbound connections to arbitrary external hosts — Spark executors have a
#   different network path than the Python driver process.
# * Foreign catalogs work because they use Unity Catalog's managed connection
#   infrastructure, not user-initiated JDBC reads.
#
# This probe implements a minimal TDS 7.x PRELOGIN + LOGIN7 handshake using raw
# sockets. It runs from the Python driver process (same process that passes the
# TCP check) and produces real SQL Server error messages on auth failure.
def app_probe_sqlserver(cfg: Dict[str, Any]) -> Dict[str, Any]:
    """
    Minimal SQL Server connectivity test via pure Python TDS protocol.
    No external libraries needed — uses raw sockets and struct packing.
    """
    import struct as _struct
    import ssl as _ssl

    host     = cfg["host"]
    port     = int(cfg.get("port") or SOURCE_PROFILES["sqlserver"]["default_port"])
    username = cfg.get("username") or ""
    password = cfg.get("password") or ""
    database = cfg.get("database") or "master"
    timeout  = int(cfg.get("timeout_seconds", 5))

    # TDS constants
    TDS_PRELOGIN = 0x12
    TDS_LOGIN7   = 0x10
    TDS_RESPONSE = 0x04
    TOKEN_ERROR    = 0xAA
    TOKEN_LOGINACK = 0xAD

    def _build_header(pkt_type, status, length):
        return _struct.pack('>BBHHBB', pkt_type, status, length, 0, 1, 0)

    def _build_prelogin(encrypt_request=0x01):
        """PRELOGIN with configurable encryption preference.
        
        encrypt_request values:
          0x01 = ENCRYPT_ON  (request encryption — default, works for RDS/Azure/most servers)
          0x02 = ENCRYPT_OFF (don't request encryption — fallback for servers without TLS)
        """
        opt_offset = 11  # 2 options (5 bytes each) + terminator
        options  = _struct.pack('>BHH', 0x00, opt_offset, 6)      # VERSION
        options += _struct.pack('>BHH', 0x01, opt_offset + 6, 1)  # ENCRYPTION
        options += b'\xFF'                                         # TERMINATOR
        data  = _struct.pack('>IH', 0x000F0000, 0x0000)           # Client version
        data += _struct.pack('B', encrypt_request)
        payload = options + data
        return _build_header(TDS_PRELOGIN, 0x01, 8 + len(payload)) + payload

    def _parse_prelogin_encryption(resp):
        """Extract the server's encryption response byte from PRELOGIN.
        
        Returns:
          0x00 = ENCRYPT_OFF (no encryption)
          0x01 = ENCRYPT_ON  (full session encryption required)
          0x02 = ENCRYPT_NOT_SUP (server does not support encryption)
          0x03 = ENCRYPT_REQ (login-only encryption required)
          None = could not parse
        """
        pos = 8  # skip TDS header
        while pos < len(resp) and resp[pos] != 0xFF:
            opt_type = resp[pos]
            opt_off = _struct.unpack_from('>H', resp, pos + 1)[0]
            opt_len = _struct.unpack_from('>H', resp, pos + 3)[0]
            if opt_type == 0x01:  # ENCRYPTION option
                return resp[8 + opt_off]
            pos += 5
        return None

    def _encrypt_password(pwd):
        """TDS password obfuscation: swap nibbles FIRST, then XOR 0xA5."""
        encoded = pwd.encode('utf-16-le')
        return bytes(((b >> 4) | ((b & 0x0F) << 4)) ^ 0xA5 for b in encoded)

    def _build_login7():
        """LOGIN7 packet with SQL Server authentication credentials."""
        hostname_b = b'D\x00B\x00X\x00'  # Fixed short hostname "DBX"
        username_b = username.encode('utf-16-le')
        password_b = _encrypt_password(password)
        appname_b  = "CDC Validator".encode('utf-16-le')
        server_b   = host.encode('utf-16-le')
        library_b  = "Python".encode('utf-16-le')
        database_b = database.encode('utf-16-le')

        # Calculate offsets after 94-byte fixed header
        offset = 94
        fields = [hostname_b, username_b, password_b, appname_b, server_b,
                  b'', library_b, b'', database_b]
        offsets_and_lens = []
        for f in fields:
            offsets_and_lens.append((offset, len(f) // 2))
            offset += len(f)

        # Fixed portion
        fixed  = _struct.pack('<I', offset)           # total length
        fixed += _struct.pack('<I', 0x730A0003)       # TDS version 7.3A
        fixed += _struct.pack('<I', 4096)             # packet size
        fixed += _struct.pack('<I', 0x07000000)       # client version
        fixed += _struct.pack('<I', 0)                 # client PID
        fixed += _struct.pack('<I', 0)                # connection ID
        fixed += _struct.pack('BBBB', 0xE0, 0x03, 0x00, 0x00)  # option flags
        fixed += _struct.pack('<I', 0)                # timezone
        fixed += _struct.pack('<I', 0x00000409)       # LCID

        # Offset/length pairs for 9 variable fields
        for off, ln in offsets_and_lens:
            fixed += _struct.pack('<HH', off, ln)

        # Client ID (6 bytes) + SSPI offset/len + AttachDB + ChangePassword + SSPILong
        fixed += b'\x00' * 6
        fixed += _struct.pack('<HH', 0, 0)            # SSPI
        fixed += _struct.pack('<HH', 0, 0)            # AttachDB
        fixed += _struct.pack('<HH', 0, 0)            # ChangePassword
        fixed += _struct.pack('<I', 0)                # SSPILong

        var_data = b''.join(fields)
        payload = fixed + var_data
        return _build_header(TDS_LOGIN7, 0x01, 8 + len(payload)) + payload

    def _parse_response(data):
        """Extract LOGINACK or ERROR tokens from TDS response."""
        if len(data) < 8:
            return _result("FAIL", "Empty response from server")
        pos = 8  # skip TDS header
        errors = []
        login_ack = False
        while pos < len(data) - 1:
            token_type = data[pos]
            pos += 1
            if token_type == TOKEN_LOGINACK:
                login_ack = True
                if pos + 2 <= len(data):
                    tlen = _struct.unpack_from('<H', data, pos)[0]
                    pos += 2 + tlen
                else:
                    break
            elif token_type == TOKEN_ERROR:
                if pos + 2 <= len(data):
                    tlen = _struct.unpack_from('<H', data, pos)[0]
                    tdata = data[pos+2:pos+2+tlen]
                    pos += 2 + tlen
                    if len(tdata) >= 8:
                        err_num = _struct.unpack_from('<I', tdata, 0)[0]
                        err_class = tdata[5]
                        msg_len = _struct.unpack_from('<H', tdata, 6)[0]
                        msg = tdata[8:8+msg_len*2].decode('utf-16-le', errors='replace')
                        errors.append(f"Error {err_num} (severity {err_class}): {msg}")
                else:
                    break
            else:
                # Skip unknown token by length prefix
                if pos + 2 <= len(data):
                    tlen = _struct.unpack_from('<H', data, pos)[0]
                    pos += 2 + tlen
                else:
                    break
        if login_ack:
            return _result("PASS", "SQL Server login successful.")
        elif errors:
            return _result("FAIL", "; ".join(errors))
        else:
            return _result("FAIL", "No LOGINACK token in server response.")

    def _tds_ssl_handshake(sock):
        """Perform TDS-wrapped TLS handshake (equivalent to sqlcmd -N -C).

        SQL Server wraps TLS records inside TDS packet headers during the
        handshake phase. After handshake completes, the TLS tunnel carries
        raw TDS packets without additional wrapping.

        Returns an (ssl_obj, incoming_bio, outgoing_bio) tuple for
        encrypting/decrypting subsequent TDS traffic.
        """
        ctx = _ssl.SSLContext(_ssl.PROTOCOL_TLS_CLIENT)
        ctx.check_hostname = False
        ctx.verify_mode = _ssl.CERT_NONE  # Equivalent to -C flag

        incoming_bio = _ssl.MemoryBIO()
        outgoing_bio = _ssl.MemoryBIO()
        ssl_obj = ctx.wrap_bio(incoming_bio, outgoing_bio, server_hostname=host)

        # Drive the TLS handshake manually, wrapping in TDS packets
        while True:
            try:
                ssl_obj.do_handshake()
                break  # Handshake complete
            except _ssl.SSLWantReadError:
                # Send any outgoing TLS data wrapped in TDS PRELOGIN packet
                tls_out = outgoing_bio.read()
                if tls_out:
                    tds_pkt = _build_header(TDS_PRELOGIN, 0x01, 8 + len(tls_out)) + tls_out
                    sock.sendall(tds_pkt)

                # Read server response, strip TDS header, feed to incoming BIO
                raw = sock.recv(32768)
                if not raw:
                    raise ConnectionError("Server closed connection during TLS handshake")
                # Strip 8-byte TDS header from each packet in the buffer
                pos = 0
                while pos < len(raw):
                    if pos + 8 > len(raw):
                        break
                    pkt_len = _struct.unpack_from('>H', raw, pos + 2)[0]
                    tls_data = raw[pos + 8:pos + pkt_len]
                    incoming_bio.write(tls_data)
                    pos += pkt_len

        # Flush any remaining outgoing data after handshake
        tls_out = outgoing_bio.read()
        if tls_out:
            tds_pkt = _build_header(TDS_PRELOGIN, 0x01, 8 + len(tls_out)) + tls_out
            sock.sendall(tds_pkt)

        return ssl_obj, incoming_bio, outgoing_bio

    # ── Execute the TDS probe ──
    start = time.monotonic()
    sock = None
    try:
        sock = socket.create_connection((host, port), timeout=timeout)

        # 1. PRELOGIN exchange — request encryption
        sock.sendall(_build_prelogin(encrypt_request=0x01))
        resp = sock.recv(4096)
        if len(resp) < 8 or resp[0] != TDS_RESPONSE:
            return _result("FAIL", "No valid TDS PRELOGIN response from server.")

        # 2. Check server's encryption response and adapt
        server_enc = _parse_prelogin_encryption(resp)
        use_tls = server_enc in (0x00, 0x01, 0x03)  # Server supports/requires encryption

        if use_tls:
            # TDS-wrapped TLS handshake (equivalent to sqlcmd -N -C)
            ssl_obj, in_bio, out_bio = _tds_ssl_handshake(sock)

            # Send LOGIN7 through the encrypted channel
            login7_pkt = _build_login7()
            ssl_obj.write(login7_pkt)
            encrypted = out_bio.read()
            sock.sendall(encrypted)

            # Read encrypted response
            raw_resp = sock.recv(32768)
            in_bio.write(raw_resp)
            decrypted = ssl_obj.read(32768)
            result = _parse_response(decrypted)
        else:
            # Server does not support encryption (ENCRYPT_NOT_SUP = 0x02)
            # Fall back: re-do PRELOGIN with ENCRYPT_OFF, then send LOGIN7 unencrypted
            sock.close()
            sock = socket.create_connection((host, port), timeout=timeout)
            sock.sendall(_build_prelogin(encrypt_request=0x02))
            resp = sock.recv(4096)
            if len(resp) < 8 or resp[0] != TDS_RESPONSE:
                return _result("FAIL", "No valid TDS PRELOGIN response (unencrypted fallback).")
            sock.sendall(_build_login7())
            resp = sock.recv(8192)
            result = _parse_response(resp)

        result["latency_ms"] = round((time.monotonic() - start) * 1000, 2)
        result["encrypted"] = use_tls
        return result

    except _ssl.SSLError as e:
        return _result("FAIL", f"TLS negotiation failed: {e}")
    except socket.timeout:
        return _result("FAIL", "Timeout during TDS handshake.")
    except ConnectionRefusedError:
        return _result("FAIL", "Connection refused during TDS probe.")
    except Exception as e:
        return _result("FAIL", f"TDS probe error: {type(e).__name__}: {e}")
    finally:
        if sock:
            try:
                sock.close()
            except Exception:
                pass


# ── PostgreSQL ──────────────────────────────────────────────────────────────────────────────────────────────
# PostgreSQL probe: use a simple connection and query to verify auth plus minimal
# query ability against the selected database.
def app_probe_postgres(cfg: Dict[str, Any]) -> Dict[str, Any]:
    """
    Minimal PostgreSQL connectivity test via psycopg2.
    Install: pip install psycopg2-binary
    """
    try:
        import psycopg2
    except ImportError:
        return _probe_unavailable("postgres")

    host     = cfg["host"]
    port     = int(cfg.get("port") or SOURCE_PROFILES["postgres"]["default_port"])
    username = cfg.get("username") or ""
    password = cfg.get("password") or ""
    # Default to the built-in `postgres` database so the probe works even when the
    # user only wants a server-level smoke test.
    database = cfg.get("database") or "postgres"
    timeout  = int(cfg.get("timeout_seconds", 5))
    sslmode  = "require" if cfg.get("tls_enabled", True) else "disable"

    try:
        start = time.monotonic()
        conn = psycopg2.connect(
            host=host, port=port,
            user=username, password=password,
            dbname=database,
            sslmode=sslmode,
            connect_timeout=timeout,
        )
        with conn:
            with conn.cursor() as cur:
                cur.execute("SELECT version(), current_user, current_database()")
                row = cur.fetchone()
        conn.close()
        latency_ms   = round((time.monotonic() - start) * 1000, 2)
        pg_ver       = (row[0] or "").split(" on ")[0] if row else "unknown"
        current_user = row[1] if row else "unknown"
        current_db   = row[2] if row else "unknown"
        return _result(
            "PASS",
            f"PostgreSQL login OK as '{current_user}' on '{current_db}'. {pg_ver}",
            latency_ms=latency_ms,
            login_user=current_user,
            database=current_db,
        )
    except psycopg2.OperationalError as e:
        msg = str(e).strip()
        if any(k in msg.lower() for k in ("password", "authentication", "pg_hba")):
            return _result("FAIL", f"PostgreSQL auth failed: {msg}. Check credentials and pg_hba.conf.")
        return _result("FAIL", f"PostgreSQL connection failed: {msg}.")
    except Exception as e:
        return _result("FAIL", f"PostgreSQL app probe error: {e}")


# ── MySQL ──────────────────────────────────────────────────────────────────────────────────────────────────────
# MySQL probe: connect and run a tiny query. A database is optional because MySQL
# can accept a connection before a default schema is selected.
def app_probe_mysql(cfg: Dict[str, Any]) -> Dict[str, Any]:
    """
    Minimal MySQL connectivity test via mysql-connector-python.
    Install: pip install mysql-connector-python
    """
    try:
        import mysql.connector
        from mysql.connector import Error as MySQLError
    except ImportError:
        return _probe_unavailable("mysql")

    host     = cfg["host"]
    port     = int(cfg.get("port") or SOURCE_PROFILES["mysql"]["default_port"])
    username = cfg.get("username") or ""
    password = cfg.get("password") or ""
    database = cfg.get("database") or None
    timeout  = int(cfg.get("timeout_seconds", 5))

    conn_args: Dict[str, Any] = dict(
        host=host, port=port,
        user=username, password=password,
        connection_timeout=timeout,
        ssl_disabled=not cfg.get("tls_enabled", True),
        use_pure=True,
    )
    if database:
        conn_args["database"] = database

    try:
        start  = time.monotonic()
        conn   = mysql.connector.connect(**conn_args)
        cursor = conn.cursor()
        cursor.execute("SELECT VERSION(), USER(), DATABASE()")
        row = cursor.fetchone()
        cursor.close()
        conn.close()
        latency_ms   = round((time.monotonic() - start) * 1000, 2)
        my_ver       = row[0] if row else "unknown"
        current_user = row[1] if row else "unknown"
        current_db   = row[2] or "(none)" if row else "unknown"
        return _result(
            "PASS",
            f"MySQL login OK as '{current_user}' on '{current_db}'. Server: {my_ver}",
            latency_ms=latency_ms,
            login_user=current_user,
            database=current_db,
        )
    except MySQLError as e:
        errno = getattr(e, "errno", 0)
        if errno == 1045:   # ER_ACCESS_DENIED_ERROR
            return _result("FAIL", f"MySQL access denied ({errno}): {e}. Check username/password and GRANT.")
        if errno in (2003, 2005, 2013):  # network errors
            return _result("FAIL", f"MySQL network error ({errno}): {e}. Check host, port, and firewall.")
        return _result("FAIL", f"MySQL error ({errno}): {e}")
    except Exception as e:
        return _result("FAIL", f"MySQL app probe error: {e}")


# ── Probe dispatcher ─────────────────────────────────────────────────────────────────────────────────────────────────
# Dispatch table keeps source-specific branching out of the main runner.
PROBE_DISPATCH = {
    "sqlserver": app_probe_sqlserver,
    "postgres":  app_probe_postgres,
    "mysql":     app_probe_mysql,
}

print("App probe functions loaded.")


In [0]:
# ── Cell 5 · Network Information ─────────────────────────────────────────────────────────────────────────────────────
# Gathers network intelligence that is critical for CDC connectivity troubleshooting:
#
# 1. Region Match      — Does the target DB region match the workspace region?
#                        Cross-region adds latency and data transfer costs.
# 2. Egress / NAT IP   — The IP this compute presents to the outside world.
#                        This is what you whitelist in security groups.
# 3. Network Path Type — Is traffic routing over public internet or a private path?
#                        Detects the most common misconfiguration: trying to reach
#                        a private-only endpoint from public compute.

import ipaddress
import re
import urllib.request


# ── Region extraction ─────────────────────────────────────────────────────────────────────────────────────────
# Patterns for extracting region from well-known cloud database hostnames.
_REGION_PATTERNS = [
    # AWS RDS / Aurora: *.region.rds.amazonaws.com
    re.compile(r"\.([a-z]{2}(?:-gov)?-[a-z]+-\d+)\.rds\.amazonaws\.com$", re.I),
    # AWS Redshift: *.region.redshift.amazonaws.com
    re.compile(r"\.([a-z]{2}(?:-gov)?-[a-z]+-\d+)\.redshift\.amazonaws\.com$", re.I),
    # Azure SQL / Postgres / MySQL: region appears in metadata, not reliably in hostname
    # Azure SQL MI: *.database.windows.net (region not in hostname)
]


def _extract_region_from_host(host: str) -> str | None:
    """Attempt to extract the cloud region from a database endpoint hostname."""
    for pattern in _REGION_PATTERNS:
        m = pattern.search(host)
        if m:
            return m.group(1)
    return None


def _get_workspace_region() -> str | None:
    """Get the current Databricks workspace region.

    Tries multiple methods in order of reliability:
    1. Spark cluster usage tag (works on classic clusters)
    2. Databricks SDK list-zones API (works on serverless)
    3. Falls back to None if unavailable
    """
    # Method 1: Spark conf (classic clusters)
    try:
        region = spark.conf.get("spark.databricks.clusterUsageTags.region", None)
        if region:
            return region
    except Exception:
        pass

    # Method 2: Databricks SDK list-zones (serverless + classic)
    try:
        from databricks.sdk import WorkspaceClient
        w = WorkspaceClient()
        zones_resp = w.api_client.do("GET", "/api/2.0/clusters/list-zones")
        default_zone = zones_resp.get("default_zone", "")
        if default_zone:
            # Zone like "us-east-2b" → region "us-east-2"
            return default_zone[:-1] if default_zone[-1].isalpha() else default_zone
    except Exception:
        pass

    return None


def region_match_check(host: str) -> Dict[str, Any]:
    """Compare the target endpoint region against the workspace region.

    Cross-region connectivity adds:
    * 20-100ms+ latency (hurts CDC replication lag)
    * Data transfer costs ($0.01-0.02/GB on AWS)
    * Potential compliance issues with data residency
    """
    target_region = _extract_region_from_host(host)
    workspace_region = _get_workspace_region()

    if not target_region:
        return _result(
            "INFO",
            f"Could not extract region from hostname '{host}'. "
            "Region matching is only supported for AWS RDS/Aurora endpoints.",
            workspace_region=workspace_region,
            target_region=None,
        )

    if not workspace_region:
        return _result(
            "INFO",
            f"Target region detected as '{target_region}' but workspace region is unavailable.",
            workspace_region=None,
            target_region=target_region,
        )

    if target_region == workspace_region:
        return _result(
            "PASS",
            f"Regions match: both in '{target_region}'. "
            "Same-region connectivity — optimal latency and no cross-region transfer costs.",
            workspace_region=workspace_region,
            target_region=target_region,
            match=True,
        )
    else:
        return _result(
            "WARN",
            f"Region mismatch: workspace is in '{workspace_region}' but target is in '{target_region}'. "
            f"Cross-region adds latency and data transfer costs (~$0.02/GB on AWS). "
            f"Consider using a Databricks workspace in '{target_region}' or setting up "
            f"a same-region replica for CDC.",
            workspace_region=workspace_region,
            target_region=target_region,
            match=False,
        )


# ── Egress / NAT IP ───────────────────────────────────────────────────────────────────────────────────────────
def get_egress_ip() -> Dict[str, Any]:
    """Discover the public IP this compute uses for outbound connections.

    This is the IP that needs to be whitelisted in:
    * AWS security group inbound rules
    * Azure SQL firewall rules
    * Any IP-based allowlist on the source database

    Uses AWS checkip as primary (fast, no rate limits) with a fallback.
    """
    endpoints = [
        "https://checkip.amazonaws.com",
        "https://ifconfig.me/ip",
    ]
    for url in endpoints:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "databricks-cdc-validator/1.0"})
            with urllib.request.urlopen(req, timeout=5) as resp:
                ip = resp.read().decode().strip()
                # Validate it looks like an IP
                ipaddress.ip_address(ip)
                return _result(
                    "INFO",
                    f"Egress IP: {ip}  — whitelist this IP in your source DB security group / firewall.",
                    egress_ip=ip,
                )
        except Exception:
            continue

    return _result(
        "WARN",
        "Could not determine egress IP. This compute may not have outbound internet access, "
        "or external IP-check services are unreachable. If using a private subnet, the egress IP "
        "is typically the NAT Gateway's Elastic IP.",
        egress_ip=None,
    )


# ── Network Path Type ─────────────────────────────────────────────────────────────────────────────────────────
def network_path_check(host: str, resolved_ips: list | None = None) -> Dict[str, Any]:
    """Determine if traffic routes over public internet or a private network path.

    Logic:
    * If the resolved IP is in a private range (RFC 1918 / RFC 6598), traffic stays
      on private infrastructure (VPC peering, PrivateLink, Transit Gateway).
    * If the resolved IP is public, traffic exits via internet gateway / NAT.

    This is the #1 diagnostic for the "Publicly Accessible = No" misconfiguration:
    if you resolve a public IP but the RDS instance is private-only, packets are
    routed to the internet and silently dropped by the VPC boundary.
    """
    if not resolved_ips:
        # Try to resolve now
        try:
            infos = socket.getaddrinfo(host, None)
            resolved_ips = sorted({item[4][0] for item in infos})
        except Exception:
            return _result(
                "INFO",
                "Cannot determine network path — DNS resolution unavailable.",
                path_type="unknown",
            )

    # Classify each resolved IP
    private_ips = []
    public_ips = []
    for ip_str in resolved_ips:
        try:
            ip_obj = ipaddress.ip_address(ip_str)
            if ip_obj.is_private:
                private_ips.append(ip_str)
            else:
                public_ips.append(ip_str)
        except ValueError:
            continue

    if private_ips and not public_ips:
        return _result(
            "PASS",
            f"Private network path detected. Resolved IPs are private: {private_ips}. "
            "Traffic stays within VPC/VNet (via peering, PrivateLink, or Transit Gateway). "
            "No internet traversal — lower latency and no public exposure.",
            path_type="private",
            private_ips=private_ips,
            public_ips=[],
        )
    elif public_ips and not private_ips:
        return _result(
            "INFO",
            f"Public network path. Resolved IPs are public: {public_ips}. "
            f"Traffic routes through internet gateway or NAT.",
            path_type="public",
            private_ips=[],
            public_ips=public_ips,
        )
    elif private_ips and public_ips:
        return _result(
            "INFO",
            f"Mixed resolution: private IPs {private_ips}, public IPs {public_ips}. "
            "This is unusual — check DNS for split-horizon or multi-homed endpoints.",
            path_type="mixed",
            private_ips=private_ips,
            public_ips=public_ips,
        )
    else:
        return _result("INFO", "No IPs to classify.", path_type="unknown")


# ── RDS Public Access Inference ────────────────────────────────────────────────────────────────────────────
def rds_public_access_check(host: str, resolved_ips: list | None = None) -> Dict[str, Any] | None:
    """Infer the RDS 'Publicly Accessible' setting from DNS resolution.

    Logic (deterministic, no false positives):
    * From OUTSIDE the RDS VPC (which is where Databricks serverless lives):
      - 'Publicly Accessible = Yes' → hostname resolves to a PUBLIC IP
      - 'Publicly Accessible = No'  → hostname resolves to a PRIVATE IP
    * This is how AWS DNS works for RDS — not a heuristic, it's by design.

    Returns None if the host is not an RDS endpoint (check not applicable).
    """
    is_rds = ".rds.amazonaws.com" in host.lower()
    if not is_rds:
        return None  # Only applicable to RDS/Aurora

    if not resolved_ips:
        return _result(
            "INFO",
            "Cannot infer 'Publicly Accessible' setting — no resolved IPs available.",
            publicly_accessible=None,
        )

    # Classify the resolved IPs
    has_public = any(not ipaddress.ip_address(ip).is_private for ip in resolved_ips)
    has_private = any(ipaddress.ip_address(ip).is_private for ip in resolved_ips)

    if has_public and not has_private:
        return _result(
            "PASS",
            f"'Publicly Accessible' is confirmed YES — RDS resolves to public IP ({', '.join(resolved_ips)}) "
            f"from outside the VPC. The instance is configured to accept external connections.",
            publicly_accessible=True,
        )
    elif has_private and not has_public:
        return _result(
            "WARN",
            f"'Publicly Accessible' appears to be NO — RDS resolves to private IP ({', '.join(resolved_ips)}) "
            f"from outside the VPC. This means connections from this compute cannot reach the instance. "
            f"Options: enable 'Publicly Accessible', set up VPC peering, or use PrivateLink.",
            publicly_accessible=False,
        )
    else:
        return _result(
            "INFO",
            f"Mixed IP resolution ({', '.join(resolved_ips)}) — cannot definitively determine "
            "'Publicly Accessible' setting. Check the RDS console.",
            publicly_accessible=None,
        )


# ── Alternative Port Probe ─────────────────────────────────────────────────────────────────────────────────
def alt_port_probe(host: str, db_port: int, timeout: int = 3) -> Dict[str, Any]:
    """Try connecting to common ports on the same host to determine if the host
    is reachable at all or if only the DB port is blocked.

    This answers the critical question:
    * "Is the host completely unreachable?" → Routing / host-level issue
    * "Is only the DB port blocked?" → Security group rule missing for that port

    If another port succeeds, the user knows for sure it’s just a missing
    inbound rule for the DB port — not a broader network issue.
    """
    # Common ports to test (likely open on most cloud DB instances)
    probe_ports = [
        (443, "HTTPS"),
        (80, "HTTP"),
    ]

    results = []
    for port, label in probe_ports:
        if port == db_port:
            continue  # Skip the DB port itself, we already tested it
        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.settimeout(timeout)
            start = time.perf_counter()
            err = sock.connect_ex((host, port))
            elapsed_ms = (time.perf_counter() - start) * 1000
            sock.close()

            if err == 0:
                results.append({"port": port, "label": label, "status": "OPEN", "latency_ms": round(elapsed_ms, 1)})
            else:
                results.append({"port": port, "label": label, "status": "CLOSED"})
        except socket.timeout:
            results.append({"port": port, "label": label, "status": "TIMEOUT"})
        except Exception as e:
            results.append({"port": port, "label": label, "status": "ERROR", "error": str(e)})

    # Interpret the results
    #
    # Important caveats for interpretation:
    # * OPEN (full handshake) = strong evidence the host is reachable on that port.
    # * CLOSED (TCP RST received) = something at that IP responded, but it could be
    #   the host kernel OR an intermediate device (load balancer, network appliance).
    #   We treat this as "likely reachable" but cannot be 100% certain.
    # * TIMEOUT (no response) = packet silently dropped. Could be Security Group,
    #   NACL, Network Firewall, or a routing blackhole. We cannot pinpoint which.
    #
    # We report observations factually and avoid blaming a specific component.

    open_ports = [r for r in results if r["status"] == "OPEN"]
    closed_ports = [r for r in results if r["status"] == "CLOSED"]
    timeout_ports = [r for r in results if r["status"] == "TIMEOUT"]

    if open_ports:
        # Strong signal: full TCP handshake completed on another port
        open_summary = ", ".join(f"{r['port']}/{r['label']}" for r in open_ports)
        return _result(
            "INFO",
            f"Other ports are open on this host ({open_summary}). "
            f"This confirms the host is reachable from this compute. "
            f"Port {db_port} appears to be specifically filtered — check inbound rules "
            f"on Security Groups, NACLs, or any Network Firewall in the path.",
            conclusion="port_specific_block",
            probed_ports=results,
        )
    elif closed_ports and not timeout_ports:
        # RST received: something responded, but we can't be 100% sure it's the host
        closed_summary = ", ".join(str(r["port"]) for r in closed_ports)
        return _result(
            "INFO",
            f"Received TCP RST (connection refused) on port(s) {closed_summary}. "
            f"This suggests traffic reaches the host (or a device in front of it) on those ports, "
            f"but port {db_port} is being silently dropped. "
            f"Most likely cause: a port-specific firewall rule (Security Group or NACL) "
            f"that doesn't include port {db_port}.",
            conclusion="likely_port_specific_block",
            probed_ports=results,
        )
    elif all(r["status"] == "TIMEOUT" for r in results):
        # All ports timeout: broader reachability issue
        probed_summary = ", ".join(f"{r['port']}/{r['label']}" for r in results)
        return _result(
            "WARN",
            f"ALL tested ports timed out ({probed_summary}). "
            f"This suggests a broader reachability issue — not specific to port {db_port}. "
            f"Possible causes: host is unreachable from this network, all ports are filtered, "
            f"'Publicly Accessible' is disabled, or the host is down.",
            conclusion="all_ports_unreachable",
            probed_ports=results,
        )
    else:
        # Mixed: some timeout, some closed, etc.
        summary_parts = []
        if closed_ports:
            summary_parts.append(f"RST on {', '.join(str(r['port']) for r in closed_ports)}")
        if timeout_ports:
            summary_parts.append(f"timeout on {', '.join(str(r['port']) for r in timeout_ports)}")
        return _result(
            "INFO",
            f"Mixed results on alternative ports: {'; '.join(summary_parts)}. "
            f"This makes it harder to isolate the issue — the filtering behavior "
            f"differs by port, which could indicate layered firewall rules.",
            conclusion="mixed",
            probed_ports=results,
        )


# ── Aggregate network info ────────────────────────────────────────────────────────────────────────────────────
def gather_network_info(host: str, resolved_ips: list | None = None, db_port: int = 5432, tcp_failed: bool = False) -> Dict[str, Any]:
    """Run all network intelligence checks and return a combined result."""
    info = {}
    info["region_match"] = region_match_check(host)
    info["egress_ip"]    = get_egress_ip()
    info["network_path"] = network_path_check(host, resolved_ips)
    # RDS-specific: infer Publicly Accessible setting from DNS
    rds_check = rds_public_access_check(host, resolved_ips)
    if rds_check is not None:
        info["rds_public_access"] = rds_check
    # Only probe alternative ports when TCP to the DB port failed — no point otherwise
    if tcp_failed:
        info["alt_port_probe"] = alt_port_probe(host, db_port)
    return info


# ── Visual Network Path Diagram ───────────────────────────────────────────────────────────────────────────────
def generate_network_diagram(result: Dict[str, Any]) -> str:
    """Generate an ASCII network path diagram showing where the connection breaks.

    The diagram adapts based on:
    * Network path type (public vs private)
    * Which check failed first (DNS, TCP, TLS, App)
    * Source type and port

    Each hop in the path is shown with a status indicator:
    * ✅ = traffic passes through this hop successfully
    * ❌ = traffic is blocked/fails at this hop
    * ❓ = unknown / not tested (downstream of a failure)
    """
    checks = result.get("checks", {})
    net_info = result.get("network_info", {})
    path_type = net_info.get("network_path", {}).get("path_type", "public")
    egress_ip = net_info.get("egress_ip", {}).get("egress_ip", "?.?.?.?")
    host = result.get("host", "target")
    port = result.get("port", "?")
    source_type = result.get("source_type", "database")

    # Short display name for the target
    target_label = source_type.upper()

    # Determine the failure point
    dns_status = checks.get("dns", {}).get("status", "SKIPPED")
    tcp_status = checks.get("tcp", {}).get("status", "SKIPPED")
    tls_status = checks.get("tls", {}).get("status", "SKIPPED")
    app_status = checks.get("app_probe", {}).get("status", "SKIPPED")

    # Build the hop list depending on path type
    # Each hop: (label, status_icon, annotation)
    hops = []

    # Hop 1: This compute
    hops.append(("This Compute", "✅", f"IP: {egress_ip}"))

    # Hop 2: Network path (varies by type)
    if path_type == "private":
        if dns_status == "PASS":
            hops.append(("VPC Peering / PrivateLink", "✅", "Private network"))
        else:
            hops.append(("VPC Peering / PrivateLink", "❓", ""))
    else:
        # Public path goes through NAT → Internet
        if dns_status == "PASS":
            hops.append(("NAT Gateway", "✅", "Outbound"))
            hops.append(("Internet", "✅", "Public path"))
        elif dns_status == "FAIL":
            hops.append(("DNS Resolution", "❌", "Cannot resolve hostname"))
            hops.append(("Internet", "❓", ""))
        else:
            hops.append(("NAT Gateway", "❓", ""))
            hops.append(("Internet", "❓", ""))

    # Hop 3: Security Group / Firewall
    if dns_status != "PASS":
        hops.append(("Security Group", "❓", f"Port {port}"))
    elif tcp_status == "FAIL":
        # Determine if it's a timeout (SG drop) or RST (port closed)
        tcp_details = checks.get("tcp", {}).get("details", "")
        if "timeout" in tcp_details.lower():
            hops.append(("Security Group", "❌", f"Dropping traffic on port {port}"))
        else:
            hops.append(("Security Group", "❌", f"Connection refused on port {port}"))
    elif tcp_status == "PASS":
        hops.append(("Security Group", "✅", f"Port {port} open"))
    else:
        hops.append(("Security Group", "❓", f"Port {port}"))

    # Hop 4: TLS layer (only if relevant)
    if tls_status == "PASS":
        hops.append(("TLS/SSL", "✅", "Encrypted"))
    elif tls_status == "FAIL":
        hops.append(("TLS/SSL", "❌", "Handshake failed"))
    elif tls_status != "SKIPPED":
        hops.append(("TLS/SSL", "❓", ""))
    # If SKIPPED (disabled or upstream failure), omit TLS from diagram

    # Hop 5: The target database
    if app_status == "PASS":
        hops.append((target_label, "✅", f"{host}:{port}"))
    elif app_status == "FAIL":
        hops.append((target_label, "❌", f"Auth/query failed"))
    elif tcp_status == "PASS" and (tls_status in ("PASS", "SKIPPED")):
        hops.append((target_label, "✅", f"{host}:{port}"))
    else:
        hops.append((target_label, "❓", f"{host}:{port}"))

    # Render the diagram
    lines = []
    lines.append("")
    lines.append("  🗺️  Connection Path")
    lines.append("")

    # Draw the flow
    flow_parts = []
    for label, icon, _ in hops:
        flow_parts.append(f"{icon} {label}")

    # Single-line flow
    flow_line = "  " + "  ──▶  ".join(flow_parts)
    lines.append(flow_line)
    lines.append("")

    # Find the failure point and add explanation
    failure_hop = None
    for i, (label, icon, annotation) in enumerate(hops):
        if icon == "❌":
            failure_hop = (label, annotation)
            break

    if failure_hop:
        lines.append(f"  ⚠️  BLOCKED AT: {failure_hop[0]}")
        if failure_hop[1]:
            lines.append(f"     Reason: {failure_hop[1]}")
    else:
        # Check if everything passed
        all_pass = all(icon in ("✅",) for _, icon, _ in hops)
        if all_pass:
            lines.append("  ✅ End-to-end path is clear!")

    lines.append("")
    return "\n".join(lines)


print("Network information functions loaded.")

In [0]:
# ── Cell 6 · Main Runner ──────────────────────────────────────────────────────────────────────────────────────────────
def run_validator(cfg: Dict[str, Any]) -> Dict[str, Any]:
    """Run the validator end to end.

    Execution order matters:
    * DNS first, because later checks depend on name resolution.
    * TCP second, because TLS and app probes are meaningless if the port is closed.
    * TLS third, because it narrows transport vs trust issues.
    * App probe last, because it is the most opinionated and requires credentials.

    The function returns one structured result object that can be printed, stored,
    or later exported as JSON.
    """
    # Validate the source type early so failures are immediate and obvious.
    source_type = cfg.get("source_type", "").strip().lower()
    if source_type not in VALID_SOURCE_TYPES:
        raise ValueError(
            f"Unsupported source_type '{source_type}'. "
            f"Must be one of: {', '.join(sorted(VALID_SOURCE_TYPES))}"
        )

    # Resolve defaults from the selected source profile.
    profile      = SOURCE_PROFILES[source_type]
    host         = cfg.get("host", "").strip()
    port         = int(cfg.get("port") or profile["default_port"])
    timeout      = int(cfg.get("timeout_seconds", 5))
    tls_enabled  = cfg.get("tls_enabled",   profile["tls_default"])
    run_app_probe = cfg.get("run_app_probe", False)

    checks: Dict[str, Any] = {}

    # 1 ─ DNS
    checks["dns"] = resolve_host(host)

    # 2 ─ TCP  (skip if DNS already failed)
    if checks["dns"]["status"] == "PASS":
        checks["tcp"] = tcp_check(host, port, timeout, resolved_ips=checks["dns"].get("ips", []))
    else:
        checks["tcp"] = _result("SKIPPED", f"Skipped — DNS check did not pass.")

    # 3 ─ TLS
    if tls_enabled:
        if checks["tcp"]["status"] == "PASS":
            # SQL Server (TDS), PostgreSQL and MySQL negotiate TLS *inside* their
            # wire protocol (STARTTLS-style), not as implicit TLS on the port. A raw
            # ssl handshake on the DB port fails even on healthy servers, so we do
            # NOT raw-handshake here. Real TLS validation happens in the app probe
            # (PG sslmode=require, MySQL ssl, SQL Server TDS-wrapped TLS).
            checks["tls"] = _result(
                "INFO",
                "TLS negotiated in-protocol by this engine; validated by the app "
                "probe (enable Run App Probe), not by a raw port handshake.",
            )
        else:
            checks["tls"] = _result(
                "SKIPPED",
                f"Skipped — TCP check status is '{checks['tcp']['status']}'. "
                "Fix TCP first.",
            )
    else:
        checks["tls"] = _result("SKIPPED", "TLS disabled by configuration.")

    # 4 ─ App probe
    if run_app_probe:
        probe_fn = PROBE_DISPATCH.get(source_type)
        if probe_fn:
            checks["app_probe"] = probe_fn({**cfg, "port": port})
        else:
            checks["app_probe"] = _result("SKIPPED", f"No probe registered for '{source_type}'.")
    else:
        checks["app_probe"] = _result("SKIPPED", "App probe disabled (run_app_probe=False).")

    # 5 ─ Network information (always runs — no dependencies on TCP/TLS success)
    resolved_ips = checks["dns"].get("ips", []) if checks["dns"]["status"] == "PASS" else None
    tcp_failed = checks.get("tcp", {}).get("status") == "FAIL"
    network_info = gather_network_info(host, resolved_ips, db_port=port, tcp_failed=tcp_failed)

    # Assemble the final machine-readable payload.
    result = {
        "source_type": source_type,
        "host":         host,
        "port":         port,
        "checks":       checks,
        "network_info": network_info,
    }
    result["final_status"]   = final_status(checks)
    result["recommendation"] = classify_recommendation(checks, host=host, port=port, network_info=network_info)
    return result


# ── Execute ────────────────────────────────────────────────────────────────────────────────────────────────────
# This is the main line to rerun after changing widget values.
result = run_validator(config)


# ──────────────────────────────────────────────────────────────────────────────────────────────────────────────
# OUTPUT — Restructured for clarity:
#   1. Header + Diagram (instant visual comprehension)
#   2. Verdict + What to do (the actionable part)
#   3. Compact check details (for those who want to dig deeper)
#   4. Network context (supporting facts)
# ──────────────────────────────────────────────────────────────────────────────────────────────────────────────

STATUS_ICON = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌", "SKIPPED": "⏭️", "INFO": "ℹ️"}
W = 64

profile  = SOURCE_PROFILES.get(result["source_type"], {})
display  = profile.get("display_name", result["source_type"])
fs_icon  = STATUS_ICON.get(result["final_status"], "?")

# ── 1. HEADER + DIAGRAM (what happened, visually) ─────────────────────────────────────────────────────────────
print("═" * W)
print(f"  Lakeflow CDC Source Validator")
print(f"  {display}  →  {result['host']}:{result['port']}")
print("═" * W)
print(generate_network_diagram(result))

# ── 2. VERDICT + WHAT TO DO ───────────────────────────────────────────────────────────────────────────────────
print("─" * W)
print(f"  {fs_icon}  RESULT: {result['final_status']}")
print("─" * W)
if result["final_status"] != "PASS":
    print(f"  {result['recommendation']}")
else:
    print("  All checks passed — connectivity looks good for CDC.")
print()

# ── 3. COMPACT CHECK DETAILS ──────────────────────────────────────────────────────────────────────────────────
print("─" * W)
print("  📋  Check Details")
print("─" * W)
checks = result["checks"]
for name, check in checks.items():
    icon = STATUS_ICON.get(check["status"], "?")
    lat  = f" [{check['latency_ms']}ms]" if "latency_ms" in check else ""
    # Shorten the detail to one readable line
    detail = check["details"]
    if len(detail) > 80:
        cut = detail.find(". ")
        if 0 < cut <= 100:
            detail = detail[:cut + 1]
        else:
            detail = detail[:77] + "..."
    print(f"  {name.upper():<11} {icon} {check['status']:<8} {detail}{lat}")
print()

# ── 4. NETWORK CONTEXT (supporting facts) ─────────────────────────────────────────────────────────────────────
print("─" * W)
print("  🌐  Network Context")
print("─" * W)

net = result.get("network_info", {})
dns_check = result.get("checks", {}).get("dns", {})
resolved = ", ".join(dns_check.get("ips", [])) or "(unresolved)"

print(f"  Host          {result['host']}")
print(f"  Resolved to   {resolved}")

# Region
reg = net.get("region_match", {})
if reg.get("match") is True:
    print(f"  Region        ✅ Both in {reg.get('workspace_region', '?')}")
elif reg.get("match") is False:
    print(f"  Region        ⚠️  Mismatch: workspace={reg.get('workspace_region')} → target={reg.get('target_region')}")
else:
    tr = reg.get("target_region") or "unknown"
    wr = reg.get("workspace_region") or "unknown"
    print(f"  Region        workspace={wr}, target={tr}")

# Egress IP
egress = net.get("egress_ip", {})
ip = egress.get("egress_ip", "unavailable")
print(f"  Egress IP     {ip}  ← whitelist this in your firewall")

# Path type
path = net.get("network_path", {})
path_type = path.get("path_type", "unknown")
path_label = {"public": "Public (internet)", "private": "Private (VPC/PrivateLink)", "mixed": "Mixed"}.get(path_type, path_type)
print(f"  Network path  {path_label}")

# RDS Access (only shown for RDS)
rds = net.get("rds_public_access", {})
if rds:
    pa = rds.get("publicly_accessible")
    if pa is True:
        print(f"  RDS access    ✅ Publicly Accessible = Yes (confirmed via DNS)")
    elif pa is False:
        print(f"  RDS access    ⚠️  Publicly Accessible = No (private IP resolved)")

# Port probe (only shown when TCP failed)
probe = net.get("alt_port_probe", {})
if probe:
    conclusion = probe.get("conclusion", "")
    if conclusion == "port_specific_block":
        print(f"  Port probe    ℹ️  Other ports open — only port {result['port']} is filtered")
    elif conclusion == "likely_port_specific_block":
        print(f"  Port probe    ℹ️  RST on other ports — port {result['port']} specifically filtered")
    elif conclusion == "all_ports_unreachable":
        print(f"  Port probe    ⚠️  ALL ports timeout — host may be fully unreachable")
    elif conclusion == "mixed":
        print(f"  Port probe    ℹ️  Mixed results across ports")

print()

# Security group / firewall quick-add (only when we have an egress IP and TCP failed)
if egress.get("egress_ip") and result["final_status"] == "FAIL":
    _qf_port = result["port"]
    _qf_ip = egress["egress_ip"]
    _qf_host = result.get("host", "").lower()
    _qf_is_rds = ".rds.amazonaws.com" in _qf_host
    _qf_is_azure = ".database.windows.net" in _qf_host or ".database.azure.com" in _qf_host

    if _qf_is_azure:
        print(f"  🛡️  Quick Fix (Azure CLI):")
        print(f"     az sql server firewall-rule create \\")
        print(f"       --resource-group <YOUR_RG> \\")
        print(f"       --server <YOUR_SERVER_NAME> \\")
        print(f"       --name AllowDatabricks \\")
        print(f"       --start-ip-address {_qf_ip} --end-ip-address {_qf_ip}")
    elif _qf_is_rds:
        print(f"  🛡️  Quick Fix (AWS CLI):")
        print(f"     aws ec2 authorize-security-group-ingress \\")
        print(f"       --group-id <YOUR_SG_ID> \\")
        print(f"       --protocol tcp --port {_qf_port} \\")
        print(f"       --cidr {_qf_ip}/32")
    else:
        print(f"  🛡️  Quick Fix:")
        print(f"     Whitelist egress IP {_qf_ip} for TCP port {_qf_port}")
        print(f"     in your database's firewall or security group inbound rules.")
    print()

print("═" * W)
